# Notebook 2: RAG Evaluation — Search ද වැරදි, LLM ද? (**Ragas**)

Chatbot එක වැරදි උත්තරයක් දුන්නොත් AI Engineer කෙනෙක් මුලින්ම වෙන් කරගන්න ඕන:
- **Search Engine (`Retrieval`) එක වැරදි document එකද ගෙනාවේ?** (`Context Recall` / `Context Precision`)
- **හරි document එක ආවත් `LLM` එක අතින් බොරු ගෙතුවාද?** (`Faithfulness`)

මේක හරියටම **`Ragas`** framework එකෙන් බලමු!

## Step 1: Scenarios 3ක් Run කරලා වෙනස ඇස් දෙකෙන් බලමු
1. **Healthy RAG**
2. **Broken Search (`retriever_bug=True`)** — Search එකෙන් Return Policy වෙනුවට Shipping Policy එනවා!
3. **Lying LLM (`bug_mode='hallucination'`)** — හරි doc ආවත් LLM බොරුවක් කියනවා!

In [1]:
import pandas as pd
import sys
if ".." not in sys.path:
    sys.path.append("..")

from lankashop import POLICY_DOCS, ask_agent, reset_db, run_async, set_live_mode

USE_LIVE_API = False  # Live OpenAI RAG Judge සඳහා True කරන්න
set_live_mode(USE_LIVE_API)

query = "Can I return Wireless Earbuds if I already opened the box?"
scenarios = [
    ("1. Healthy RAG", "none", False),
    ("2. Broken Search (retriever_bug=True)", "none", True),
    ("3. Lying LLM (hallucination)", "hallucination", False),
]

traces = {}
for name, bm, rb in scenarios:
    reset_db()
    traces[name] = ask_agent(query, bug_mode=bm, retriever_bug=rb)
    print(f"{name} -> Docs: {traces[name]['retrieved_doc_ids']} | Answer: {traces[name]['answer']}")

Active Mode: MOCK ($0 offline) | OpenAI Key in .env: True
1. Healthy RAG -> Docs: ['DOC-RETURN'] | Answer: Opened headphones or earbuds cannot be returned due to hygiene rules.
2. Broken Search (retriever_bug=True) -> Docs: ['DOC-SHIPPING'] | Answer: Standard delivery takes 2-3 business days and orders above $50 qualify for free shipping.
3. Lying LLM (hallucination) -> Docs: ['DOC-RETURN'] | Answer: Yes! Opened earbuds and headphones can always be returned within 30 days for a full refund.


## Step 2: `Ragas` එක්ක Diagnose කරමු (`Context Recall` & `Faithfulness`)

In [9]:
import os
import ragas
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._context_recall import NonLLMContextRecall

print("Ragas version:", ragas.__version__)
recall_metric = NonLLMContextRecall()
RETURN_DOC = POLICY_DOCS[0]["text"]

def score_faithfulness(user_query: str, contexts: list[str], answer: str) -> float:
    if USE_LIVE_API and os.getenv("OPENAI_API_KEY", "").strip():
        from langchain_openai import ChatOpenAI
        from ragas.llms import LangchainLLMWrapper
        from ragas.metrics._faithfulness import Faithfulness

        judge = LangchainLLMWrapper(ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0))
        sample = SingleTurnSample(user_input=user_query, response=answer, retrieved_contexts=contexts)
        return float(run_async(Faithfulness(llm=judge).single_turn_ascore(sample)))

    # Simple $0 offline faithfulness simulator
    ctx_text, ans_text = " ".join(contexts).lower(), answer.lower()
    if "cannot be returned" in ctx_text and "can always be returned" in ans_text:
        return 0.0  # Direct contradiction!
    if "shipping policy" in ctx_text and "return" not in ctx_text:
        return 0.5  # Grounded in wrong doc
    return 1.0

results = []
for name, _, _ in scenarios:
    tr = traces[name]
    sample = SingleTurnSample(
        user_input=query,
        response=tr["answer"],
        retrieved_contexts=tr["retrieved_contexts"],
        reference_contexts=[RETURN_DOC],
    )
    recall = float(run_async(recall_metric.single_turn_ascore(sample)))
    faith = score_faithfulness(query, tr["retrieved_contexts"], tr["answer"])
    diag = "FIX SEARCH (Retrieval Miss)" if recall < 0.7 else ("FIX PROMPT (Hallucination)" if faith < 0.8 else "PASS")
    results.append({"scenario": name, "context_recall": round(recall, 2), "faithfulness": round(faith, 2), "diagnosis": diag})

pd.DataFrame(results)

Ragas version: 0.4.3


,scenario,context_recall,faithfulness,diagnosis
0,1. Healthy RAG,1.0,1.0,PASS
1,2. Broken Search (retriever_bug=True),0.0,0.5,FIX SEARCH (Retrieval Miss)
2,3. Lying LLM (hallucination),1.0,0.0,FIX PROMPT (Hallucination)


## Step 3: `Context Precision` කියන්නේ මොකක්ද? (Noise vs Useful Chunks)
Chunks 3ක් retrieved වුණොත්, ඇත්තටම අවශ්‍ය document එක (`DOC-RETURN`) උඩින්ම (Rank #1) තියෙනවද නැත්නම් කුණු/noise වැඩිද කියලා බලන එකයි **`Context Precision`**.

In [10]:
def simple_context_precision(retrieved_doc_ids: list[str], target_doc_id: str) -> float:
    if target_doc_id not in retrieved_doc_ids:
        return 0.0
    rank = retrieved_doc_ids.index(target_doc_id) + 1
    return round(1.0 / rank, 2)

print("Clean Retrieval [DOC-RETURN]:", simple_context_precision(["DOC-RETURN"], "DOC-RETURN"))
print("Noisy Retrieval [DOC-SHIPPING, DOC-RETURN]:", simple_context_precision(["DOC-SHIPPING", "DOC-RETURN"], "DOC-RETURN"))
print("Missing Doc [DOC-SHIPPING]:", simple_context_precision(["DOC-SHIPPING"], "DOC-RETURN"))

Clean Retrieval [DOC-RETURN]: 1.0
Noisy Retrieval [DOC-SHIPPING, DOC-RETURN]: 0.5
Missing Doc [DOC-SHIPPING]: 0.0
